# AiiDA calcfunctions, workfunctions, QueryBuilder, and extras

This notebook introduces a compact AiiDA workflow for teaching purposes.

The goals are:

1. understand the difference between a normal Python function and an AiiDA `calcfunction`;
2. run a meaningful AiiDA `workfunction` that orchestrates calcfunctions;
3. inspect the process nodes created by AiiDA;
4. use the `QueryBuilder` to find `CalcFunctionNode` objects;
5. attach an extra field to one node;
6. use `QueryBuilder` filters to find only nodes carrying that extra.

The examples are intentionally small. The point is not the calculation itself, but the provenance graph that AiiDA creates around it.


## 0. Load the AiiDA profile

The next cell loads the AiiDA Jupyter extension and activates the default AiiDA profile.

After this cell has run, the notebook can use AiiDA objects such as `orm.Int`, `orm.Str`, `CalcFunctionNode`, `WorkFunctionNode`, and `QueryBuilder`.


In [ ]:
%load_ext aiida
%aiida


## 1. Imports and helper functions

We import the AiiDA ORM (`orm`) and engine (`engine`).

We also import `LinkType`, which is useful for identifying which process created or returned a given output node.


In [ ]:
from aiida import orm, engine

try:
    from aiida.common import LinkType
except ImportError:
    # Compatibility with older AiiDA versions
    from aiida.common.links import LinkType


def extras_dict(node):
    """Return all extras of a node as a regular Python dictionary."""
    try:
        all_extras = node.base.extras.all
        if callable(all_extras):
            all_extras = all_extras()
        return dict(all_extras)
    except AttributeError:
        return dict(node.extras)


## 2. Start from a normal Python function

This is an ordinary Python function. It takes a number and a category name, and returns a labelled string.

AiiDA does **not** automatically store provenance for normal Python functions. If we run this function, AiiDA will not know that it happened.


In [ ]:
def make_labelled_text_plain(number, category):
    """Return a string obtained by joining a number and a category name."""
    return f"{number}_{category}"


make_labelled_text_plain(34, "Cars")


## 3. Turn the function into an AiiDA calcfunction

A `calcfunction` is a lightweight AiiDA process.

The important differences with respect to a normal Python function are:

- the inputs should be AiiDA data nodes, for example `orm.Int` and `orm.Str`;
- the output should also be an AiiDA data node;
- AiiDA stores the process node and the provenance links between inputs, process, and output.

A `calcfunction` is appropriate for deterministic operations that create new data from given inputs.


In [ ]:
@engine.calcfunction
def make_labelled_text(number: orm.Int, category: orm.Str) -> orm.Str:
    """Join an AiiDA Int and an AiiDA Str into a new AiiDA Str."""
    return orm.Str(f"{number.value}_{category.value}")


## 4. Run the calcfunction directly

When the calcfunction is called, AiiDA stores:

- the input nodes;
- the output node;
- the `CalcFunctionNode` that connects inputs and output.

The variable `direct_labelled_text` is the output `orm.Str` node.


In [ ]:
direct_labelled_text = make_labelled_text(orm.Int(34), orm.Str("Cars"))

print("Output node:", direct_labelled_text)
print("Output value:", direct_labelled_text.value)


## 5. Find the CalcFunctionNode that created the result

The output node was created by a process node. For a calcfunction, this process node is a `CalcFunctionNode`.

The output node has an incoming `CREATE` link from the process node.


In [ ]:
direct_calcfunction_node = direct_labelled_text.base.links.get_incoming(
    link_type=LinkType.CREATE
).one().node

print("CalcFunctionNode:", direct_calcfunction_node)
print("PK:", direct_calcfunction_node.pk)
print("Process label:", direct_calcfunction_node.process_label)


## 6. Inspect the calcfunction process from the command line

The same process node can also be inspected with `verdi`.

This is useful because students should learn to connect what they see in Python with the command-line tools.


In [ ]:
!verdi process show {direct_calcfunction_node.pk}


## 7. Add a second calcfunction

The next calcfunction does something different: it computes the number of characters in an AiiDA `Str` node and returns an AiiDA `Int` node.

This gives us a better workfunction example. The workfunction will not simply duplicate the first calcfunction; instead, it will orchestrate a small two-step workflow.


In [ ]:
@engine.calcfunction
def count_characters(text: orm.Str) -> orm.Int:
    """Return the number of characters in an AiiDA Str."""
    return orm.Int(len(text.value))


## 8. Define a workfunction that uses calcfunctions

A `workfunction` is another lightweight AiiDA process.

A useful way to explain the difference is:

- a `calcfunction` creates new data from inputs;
- a `workfunction` orchestrates several steps and returns results.

Here the workfunction calls two calcfunctions:

1. `make_labelled_text` creates a labelled text such as `35_Cars`;
2. `count_characters` computes the length of that labelled text.

This is deliberately more informative than a workfunction that simply repeats exactly the same operation as the calcfunction.


In [ ]:
@engine.workfunction
def analyse_category(number: orm.Int, category: orm.Str):
    """Create a labelled text and compute its length using calcfunctions."""
    labelled_text = make_labelled_text(number, category)
    n_characters = count_characters(labelled_text)

    return {
        "labelled_text": labelled_text,
        "n_characters": n_characters,
    }


## 9. Execute the workfunction

The output `workfunction_outputs` is a dictionary-like object containing the output nodes returned by the workfunction.

The key point for students is that running this workfunction creates:

- one `WorkFunctionNode` for the orchestration step;
- one `CalcFunctionNode` for `make_labelled_text`;
- one `CalcFunctionNode` for `count_characters`.


In [ ]:
workfunction_outputs = analyse_category(orm.Int(35), orm.Str("Cars"))

workfunction_labelled_text = workfunction_outputs["labelled_text"]
workfunction_n_characters = workfunction_outputs["n_characters"]

print("Workfunction outputs:", workfunction_outputs)
print("Labelled text node:", workfunction_labelled_text)
print("Labelled text value:", workfunction_labelled_text.value)
print("Number of characters node:", workfunction_n_characters)
print("Number of characters value:", workfunction_n_characters.value)


## 10. Find and inspect the WorkFunctionNode

The workfunction returns output nodes through `RETURN` links.

We can use one of these output nodes to find the `WorkFunctionNode` that returned it.

Notice the distinction:

- the `labelled_text` node was **created** by a `CalcFunctionNode`;
- the same `labelled_text` node was **returned** by a `WorkFunctionNode`.


In [ ]:
workfunction_node = workfunction_labelled_text.base.links.get_incoming(
    link_type=LinkType.RETURN
).one().node

internal_calcfunction_node = workfunction_labelled_text.base.links.get_incoming(
    link_type=LinkType.CREATE
).one().node

print("WorkFunctionNode that returned the labelled text:")
print("  node:", workfunction_node)
print("  PK:", workfunction_node.pk)
print("  process label:", workfunction_node.process_label)

print("\nCalcFunctionNode that created the labelled text:")
print("  node:", internal_calcfunction_node)
print("  PK:", internal_calcfunction_node.pk)
print("  process label:", internal_calcfunction_node.process_label)


## 11. Inspect the workfunction process from the command line

This should show a `WorkFunctionNode`. Compare it with the `CalcFunctionNode` inspected earlier.


In [ ]:
!verdi process show {workfunction_node.pk}


## 12. Inspect the calcfunction called inside the workfunction

The workfunction called `make_labelled_text`, and that call created its own `CalcFunctionNode`.

This is useful for teaching provenance: the workflow-level node and the calculation-level node are different nodes in the graph.


In [ ]:
!verdi process show {internal_calcfunction_node.pk}


## 13. Use QueryBuilder to find CalcFunctionNode objects

The `QueryBuilder` searches the AiiDA database.

In the next cell we query all `CalcFunctionNode` objects and order them from newest to oldest.

If you ran the direct calcfunction and the workfunction above, you should see recent `CalcFunctionNode` objects such as:

- the direct call to `make_labelled_text`;
- the call to `make_labelled_text` inside the workfunction;
- the call to `count_characters` inside the workfunction.


In [ ]:
qb = orm.QueryBuilder()
qb.append(orm.CalcFunctionNode, tag="calcfunction")
qb.order_by({"calcfunction": {"ctime": "desc"}})
qb.limit(10)

calcfunction_nodes = qb.all(flat=True)

for node in calcfunction_nodes:
    print(
        f"PK={node.pk:>6}  label={node.process_label:<30}  "
        f"ctime={node.ctime}  extras={extras_dict(node)}"
    )


## 14. Set an extra on one CalcFunctionNode

Extras are user-defined metadata fields that can be attached to AiiDA nodes.

Here we attach the extra field `test` to the `CalcFunctionNode` that was created by the first direct calcfunction call.

We set:

```python
test = 2
```

This is not part of the scientific provenance. It is metadata that we can use later for searching, tagging, or teaching exercises.


In [ ]:
node_to_mark = direct_calcfunction_node

node_to_mark.base.extras.set("test", 2)

print(f"Set extra 'test' = {extras_dict(node_to_mark).get('test')} on CalcFunctionNode<{node_to_mark.pk}>")
print("All extras:", extras_dict(node_to_mark))


## 15. Query only CalcFunctionNode objects that have the extra `test`

Now we use a QueryBuilder filter.

The filter

```python
{"extras": {"has_key": "test"}}
```

means: return only nodes whose extras contain a key called `test`.


In [ ]:
qb = orm.QueryBuilder()
qb.append(
    orm.CalcFunctionNode,
    tag="calcfunction",
    filters={"extras": {"has_key": "test"}},
)
qb.order_by({"calcfunction": {"ctime": "desc"}})

nodes_with_test = qb.all(flat=True)

for node in nodes_with_test:
    print(
        f"PK={node.pk:>6}  label={node.process_label:<30}  "
        f"test={extras_dict(node).get('test')}"
    )


## 16. Query only CalcFunctionNode objects where `test == 2`

We can also filter by the value of the extra.

The filter

```python
{"extras.test": 2}
```

means: return only nodes whose extra field `test` has value `2`.


In [ ]:
qb = orm.QueryBuilder()
qb.append(
    orm.CalcFunctionNode,
    tag="calcfunction",
    filters={"extras.test": 2},
)
qb.order_by({"calcfunction": {"ctime": "desc"}})

nodes_with_test_equal_2 = qb.all(flat=True)

for node in nodes_with_test_equal_2:
    print(
        f"PK={node.pk:>6}  label={node.process_label:<30}  "
        f"test={extras_dict(node).get('test')}"
    )


## 17. Summary

In this notebook we have seen that:

- a normal Python function runs without AiiDA provenance;
- a `calcfunction` creates a `CalcFunctionNode` and stores provenance for inputs and outputs;
- a `workfunction` creates a `WorkFunctionNode` and can orchestrate several AiiDA steps;
- a workfunction can return nodes created by calcfunctions;
- `QueryBuilder` can retrieve process nodes from the database;
- extras are convenient metadata fields;
- filters on extras make it possible to retrieve only selected nodes.
